# 🚀 Introdução ao Kimi via API

Este notebook apresenta recursos iniciais do **Kimi** (Moonshot AI) consumidos via API, mostrando o quão poderoso pode ser o modelo em tarefas de linguagem, visão, ferramentas e geração estruturada.

**Modelo usado:** `kimi-k3` (ou `kimi-k2-0711-preview`, conforme disponibilidade)

**Documentação:** https://platform.moonshot.cn/docs

## 1. Instalação e configuração

In [ ]:
# Instale a biblioteca oficial (OpenAI-compatible)
!pip install -q openai python-dotenv

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("KIMI_API_KEY"),
    base_url="https://api.moonshot.ai/v1"
)

MODEL = "kimi-k3"

## 2. Chat completion simples

O básico: envie uma lista de mensagens e receba uma resposta.

## 3. Respostas em streaming

O streaming permite mostrar texto aos poucos, ideal para interfaces interativas.

In [3]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Escreva um parágrafo sobre inteligência artificial generativa."}],
    stream=True,
)

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

A inteligência artificial generativa é um ramo da IA dedicado à criação de conteúdos novos e originais, como textos, imagens, músicas, vídeos e códigos de programação. Diferentemente dos sistemas tradicionais, que se limitam a classificar ou analisar dados existentes, os modelos generativos aprendem padrões a partir de grandes volumes de informação e utilizam esse conhecimento para produzir resultados inéditos. Tecnologias como grandes modelos de linguagem (LLMs), redes neurais profundas e arquiteturas como transformers e modelos de difusão estão na base dessa capacidade criativa. Essa tecnologia tem transformado diversos setores — da educação à saúde, do entretenimento ao desenvolvimento de software — aumentando a produtividade e ampliando as possibilidades criativas de profissionais e empresas. Ao mesmo tempo, a IA generativa levanta questões importantes sobre direitos autorais, desinformação, vieses algorítmicos e o futuro do trabalho, o que torna essencial o debate sobre seu desenv

## 4. System prompt e controle de comportamento

O system prompt define a persona, formato e restrições do modelo.

In [4]:
system_prompt = """
Você é um tradutor técnico sênior. Regras:
1. Traduza do português para o inglês técnico.
2. Mantenha termos de código e APIs em inglês.
3. Responda APENAS com a tradução, sem explicações.
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "O método __init__ é o construtor da classe e recebe self como primeiro argumento."}
]

print(chat(messages))

The __init__ method is the class constructor and receives self as its first argument.


## 5. Function Calling (tools)

O modelo pode decidir chamar funções externas para obter dados em tempo real. Aqui simulamos uma consulta de previsão do tempo.

In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Obtém a previsão do tempo para uma cidade.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "Nome da cidade"},
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Unidade de temperatura"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

messages = [
    {"role": "user", "content": "Qual a temperatura em São Paulo agora?"}
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message
print(message)

ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='get_weather_0', function=Function(arguments='{"city":"São Paulo","unit":"celsius"}', name='get_weather'), type='function', index=0)], reasoning_content='The user is asking for the current temperature in São Paulo. I should use the get_weather function to get this information. Since the user is asking in Portuguese, they likely want the temperature in Celsius.')


In [6]:
# Simula o resultado da função e devolve ao modelo
def get_weather(city, unit="celsius"):
    return {"city": city, "temperature": 22, "unit": unit, "condition": "parcialmente nublado"}

if message.tool_calls:
    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    arguments = eval(tool_call.function.arguments)  # em produção, use json.loads
    
    if function_name == "get_weather":
        result = get_weather(**arguments)
    
    messages.append({
        "role": "assistant",
        "content": None,
        "tool_calls": [tool_call.model_dump()]
    })
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": str(result)
    })
    
    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages
    )
    print(final_response.choices[0].message.content)

A temperatura em **São Paulo** agora é de **22°C**, com o céu **parcialmente nublado**. 🌤️

Uma temperatura bem agradável! Precisa de mais alguma informação?


## 6. JSON Mode / saída estruturada

Force o modelo a responder em JSON válido, útil para integração com código.

In [7]:
import json

messages = [
    {"role": "system", "content": "Você é um analisador de produtos. Responda SEMPRE em JSON válido."},
    {
        "role": "user",
        "content": """
Extraia os pontos positivos e negativos do produto descrito abaixo.

Produto: Smartphone KimiPhone Pro
"A câmera é excelente em pouca luz, mas a bateria dura pouco mais de meio dia com uso intenso. O desempenho é fluido e a tela tem cores vibrantes."

Formato esperado:
{
  "produto": "string",
  "pontos_positivos": ["string"],
  "pontos_negativos": ["string"],
  "nota_geral": number
}
"""
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    response_format={"type": "json_object"}
)

data = json.loads(response.choices[0].message.content)
print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "produto": "Smartphone KimiPhone Pro",
  "pontos_positivos": [
    "Câmera excelente em pouca luz",
    "Desempenho fluido",
    "Tela com cores vibrantes"
  ],
  "pontos_negativos": [
    "Bateria dura pouco mais de meio dia com uso intenso"
  ],
  "nota_geral": 3.8
}


## 8. Conversa multi-turn com memória

Mantenha o histórico de mensagens para criar diálogos coerentes.

In [8]:
conversation = [
    {"role": "system", "content": "Você é um tutor de programação paciente."}
]

def ask(question):
    conversation.append({"role": "user", "content": question})
    answer = chat(conversation)
    conversation.append({"role": "assistant", "content": answer})
    return answer

print("Turno 1:", ask("O que é uma API REST?"))
print("\nTurno 2:", ask("E qual a diferença para GraphQL?"))
print("\nTurno 3:", ask("Me dê um exemplo de quando escolher cada uma?"))

Turno 1: # O que é uma API REST?

Ótima pergunta! Vamos por partes para ficar bem claro. 😊

## Primeiro: o que é uma API?

**API** (Application Programming Interface) é uma forma de dois sistemas conversarem entre si. 

Pense num restaurante: você (o cliente) não entra na cozinha para pegar a comida. Você faz o pedido ao **garçom**, ele leva o pedido à cozinha e traz o resultado. A API é esse garçom — ela recebe pedidos de um programa e devolve respostas de outro sistema.

## E o que é REST?

**REST** (Representational State Transfer) é um **conjunto de regras e boas práticas** para construir APIs usando o protocolo HTTP — o mesmo protocolo que a web usa.

Uma API REST funciona basicamente assim:

1. O cliente faz uma **requisição** para uma URL (chamada de *endpoint*)
2. O servidor processa e devolve uma **resposta**, geralmente em formato **JSON**

## Os métodos HTTP

A mágica do REST está em usar os verbos HTTP para indicar a **ação** desejada:

| Método | Ação | Exemplo |
|--------

## 9. Chain-of-Thought: resolvendo problemas passo a passo

Incentive o modelo a mostrar o raciocínio antes da resposta final.

In [9]:
messages = [
    {
        "role": "user",
        "content": """
Resolva passo a passo:
Um trem parte de A para B a 60 km/h. Outro parte de B para A a 80 km/h.
A distância entre A e B é 280 km. Em quanto tempo eles se encontram?
Mostre o raciocínio completo antes da resposta final.
"""
    }
]

print(chat(messages, temperature=1.0))

# Resolução Passo a Passo

## Passo 1: Entender a situação

- **Trem 1:** parte de A em direção a B a 60 km/h
- **Trem 2:** parte de B em direção a A a 80 km/h
- **Distância total:** 280 km
- Os trens se movem **um em direção ao outro** (sentidos opostos)

## Passo 2: Montar as equações de posição

Vamos medir a posição a partir do ponto A. Chamando de *t* o tempo (em horas) até o encontro:

- **Posição do Trem 1** (sai de A): 
$$x_1 = 60t$$

- **Posição do Trem 2** (sai de B, que está a 280 km de A): 
$$x_2 = 280 - 80t$$

## Passo 3: Condição de encontro

No momento do encontro, os dois trens estão na **mesma posição**:

$$x_1 = x_2$$

$$60t = 280 - 80t$$

## Passo 4: Resolver a equação

$$60t + 80t = 280$$

$$140t = 280$$

$$t = \frac{280}{140} = 2 \text{ horas}$$

> **Atalho conceitual:** como os trens se aproximam um do outro, a *velocidade relativa* é a soma das velocidades: 60 + 80 = 140 km/h. Então: tempo = 280 ÷ 140 = 2 h.

## Passo 5: Verificação

- Trem 1 percorreu: 60 × 2 = 

## 11. Análise de tokens e custo estimado

A API retorna informações de uso, permitindo estimar custos.

In [12]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Resuma a Revolução Industrial em 3 frases."}]
)

usage = response.usage
print(f"Prompt tokens: {usage.prompt_tokens}")
print(f"Completion tokens: {usage.completion_tokens}")
print(f"Total tokens: {usage.total_tokens}")

Prompt tokens: 98
Completion tokens: 533
Total tokens: 631


## 12. Requisições assíncronas (opcional)

Use a versão assíncrona do cliente para chamadas concorrentes.

In [13]:
import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(
    api_key=os.getenv("KIMI_API_KEY"),
    base_url="https://api.moonshot.ai/v1"
)

async def perguntar(assunto):
    r = await async_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"Explique {assunto} em uma frase."}]
    )
    return r.choices[0].message.content

resultados = await asyncio.gather(
    perguntar("machine learning"),
    perguntar("blockchain"),
    perguntar("computação em nuvem")
)

for r in resultados:
    print("-", r)

- Machine learning é uma área da inteligência artificial na qual computadores aprendem padrões a partir de dados, permitindo-lhes fazer previsões ou tomar decisões sem serem explicitamente programados para cada situação específica.
- Blockchain é um registro digital distribuído e imutável que armazena transações em blocos encadeados e protegidos por criptografia, permitindo que múltiplas partes compartilhem informações confiáveis sem a necessidade de um intermediário central.
- Computação em nuvem é a entrega de serviços de computação — como armazenamento, processamento e software — pela internet, permitindo que você acesse recursos sob demanda sem precisar manter infraestrutura física própria.


## Próximos passos

- Experimente prompts mais complexos para seu domínio.
- Combine function calling com JSON mode para agents.
- Implemente RAG (Retrieval-Augmented Generation) usando embeddings.
- Avalie diferentes valores de `temperature` e `top_p`.
- Consulte a documentação oficial para limites de contexto e preços.